In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import json
import os
import gc
# import mediapipe as mp
# mp_holistic = mp.solutions.holistic
from tensorflow.keras import layers, models, optimizers

In [ ]:
# Prevent OpenCV from competing with TensorFlow's multi-threading,
# which can cause graph execution errors in tf.data.Datasets
cv2.setNumThreads(0)

# A predefined subset of 35 critical facial landmarks (eyes, eyebrows, mouth)
# This reduces the face feature bloat from 1404 values down to 105 values.
SELECTED_FACE_INDICES = [
    # Lips
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95,
    # Left Eye & Eyebrow
    33, 133, 159, 145, 46, 52, 53,
    # Right Eye & Eyebrow
    362, 263, 386, 374, 276, 282, 283
]

def extract_keypoints(results):
    """Extracts, normalizes, and flattens landmarks from MediaPipe Holistic results."""

    # POINT B: Define an anchor point for spatial normalization.
    # We use the Pose Nose (landmark 0) if available.
    if results.pose_landmarks:
        anchor_x = results.pose_landmarks.landmark[0].x
        anchor_y = results.pose_landmarks.landmark[0].y
        anchor_z = results.pose_landmarks.landmark[0].z
    else:
        # Fallback if no pose is detected at all
        anchor_x, anchor_y, anchor_z = 0.0, 0.0, 0.0

    # Pose: 33 landmarks. Normalize x, y, z, but keep visibility untouched.
    if results.pose_landmarks:
        pose = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z, res.visibility]
                         for res in results.pose_landmarks.landmark]).flatten()
    else:
        pose = np.zeros(33 * 4)

    # Face (POINT A): Only extract the 35 indices defined above and normalize them.
    if results.face_landmarks:
        face = np.array([[results.face_landmarks.landmark[i].x - anchor_x,
                          results.face_landmarks.landmark[i].y - anchor_y,
                          results.face_landmarks.landmark[i].z - anchor_z]
                         for i in SELECTED_FACE_INDICES]).flatten()
    else:
        face = np.zeros(len(SELECTED_FACE_INDICES) * 3)

    # Left Hand: 21 landmarks. Normalize x, y, z.
    if results.left_hand_landmarks:
        lh = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z]
                       for res in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(21 * 3)

    # Right Hand: 21 landmarks. Normalize x, y, z.
    if results.right_hand_landmarks:
        rh = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z]
                       for res in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(21 * 3)

    return np.concatenate([pose, face, lh, rh])

In [ ]:
def npy_data_generator(precompute_dir, metadata_dir, video_ids):
    """
    Step 2: Lightning-fast generator that loads precomputed .npy files during training.
    """
    for video_id in video_ids:
        npy_path = os.path.join(precompute_dir, f"{video_id}.npy")
        json_path = os.path.join(metadata_dir, f"{video_id}.json")

        if not os.path.exists(npy_path) or not os.path.exists(json_path):
            continue

        with open(json_path, 'r') as f:
            meta = json.load(f)
            label = int(meta['label'])

        data = np.load(npy_path)
        yield data, np.int32(label)

In [ ]:
def build_model(input_shape, num_classes):
    """
    Mask-Safe Architecture: Projection + Bidirectional LSTM
    """
    inputs = layers.Input(shape=input_shape)

    # 1. Masking
    # This officially flags all -99.0 frames to be ignored by compatible layers
    x = layers.Masking(mask_value=-99.0)(inputs)

    # 2. Spatial Projection (Safe for masks)
    # Instead of a CNN, we use a Dense layer to find relationships between the 363
    # coordinates in each individual frame, expanding them to 256 features.
    x = layers.Dense(256, activation='relu')(x)
    # We use LayerNormalization instead of BatchNormalization.
    # LayerNorm normalizes per-frame, not per-batch, so padding doesn't corrupt it.
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    # 3. Temporal Modeling (Safe for masks)
    # LSTMs natively understand the Keras Masking layer. They will completely
    # stop processing when they hit the padded frames.
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False))(x)

    # 4. Classification Head
    x = layers.Dense(256, activation='relu')(x)

    x = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=x)

In [ ]:
if __name__ == "__main__":
    # Settings
    VIDEO_PATH = "/content/drive/MyDrive/wlasl_train_data"
    METADATA_PATH = "/content/drive/MyDrive/instance_metadata"
    PRECOMPUTED_PATH = "/content/drive/MyDrive/wlasl_precomputed"
    NUM_CLASSES = 100
    BATCH_SIZE = 16

    # --- NEW: Explicitly define the 1D input shape ---
    # None = variable number of frames. 363 = number of flattened landmarks.
    INPUT_SHAPE = (None, 363)

    # Prepare IDs
    if os.path.exists(METADATA_PATH):
        json_files = [f for f in os.listdir(METADATA_PATH) if f.endswith('.json')]
        video_ids = [f.replace('.json', '') for f in json_files]
    else:
        video_ids = []

    np.random.seed(42)
    np.random.shuffle(video_ids)

    # --- STEP 1: Precompute Data ---
    # precompute_dataset(VIDEO_PATH, METADATA_PATH, PRECOMPUTED_PATH, video_ids)

    split = int(len(video_ids) * 0.8)
    train_ids, val_ids = video_ids[:split], video_ids[split:]

    # --- STEP 2: TF Dataset Pipeline ---
    def get_dataset(ids, is_train=True):
        ds = tf.data.Dataset.from_generator(
            lambda: npy_data_generator(PRECOMPUTED_PATH, METADATA_PATH, ids),
            output_signature=(
                tf.TensorSpec(shape=INPUT_SHAPE, dtype=tf.float32),
                tf.TensorSpec(shape=(), dtype=tf.int32)
            )
        )
        if is_train:
            ds = ds.shuffle(100)

        # FIX: Pad features with -99.0, and pad labels with 0 (labels don't matter for padding)
        return ds.padded_batch(
            BATCH_SIZE,
            padding_values=(-99.0, 0)
        ).prefetch(tf.data.AUTOTUNE)

    train_ds = get_dataset(train_ids)
    val_ds = get_dataset(val_ids, is_train=False)

    # Build and Compile
    model = build_model(INPUT_SHAPE, NUM_CLASSES)
    model.compile(
        # --- CHANGED: Lowered learning rate for Transformer stability ---
        optimizer=optimizers.Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"Training on batches of {BATCH_SIZE}...")
    if len(train_ids) > 0:
      # for inputs, labels in train_ds.take(1):
      #     print("Input shape:", inputs.shape)
      #     print("Input min/max:", tf.reduce_min(inputs).numpy(), tf.reduce_max(inputs).numpy())
      #     print("Labels:", labels.numpy())
      #     break
      model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=100,
            verbose=1
      )
        # --- CHANGED: Updated filename ---
    model.save("wlasl_mediapipe_transformer_model.keras")